In [1]:
from dotenv import load_dotenv
import getpass
import os

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()
if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

True

### Chat models selection

Qwen3-4B-Instruct-2507

In [12]:
import importlib
import eval_utils

importlib.reload(eval_utils)

<module 'eval_utils' from '/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py'>

In [13]:
from eval_utils import load_llm, load_embeddings, load_docs

# load chat model
chat_model = load_llm(model_id="Qwen/Qwen3-4B-Instruct-2507",
                      model_path="./hf_models/Qwen3-4B-Instruct-2507")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

# load docs
docs = load_docs("data/mk4s_manuel")


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.97it/s]
Device set to use cuda:0
100%|██████████| 8/8 [00:00<00:00, 9600.70it/s]


In [14]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, 
                                               chunk_overlap=32,
                                               separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                               is_separator_regex=False,
                                               keep_separator=False,)
chunks = text_splitter.split_documents(docs)


Initialize Hybrid Retriever

In [15]:
import time
from typing import List, Set
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_core.retrievers import BaseRetriever
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.runnables import RunnableLambda
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document

# set hybrid retriever as base retriever
vector_storage = FAISS.from_documents(chunks, embedding)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

faiss_retriever = vector_storage.as_retriever(
                                search_type="mmr",
                                search_kwargs={"k": 10, "lambda_mult": 0.7}
                                )
    
hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever], weights=[0.4, 0.6]
    )

Initialize Multi-query retriever

In [16]:
mq_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        """ You are a helpful research assistant. 
        Given the user's question below, generate **exactly three** alternative search queries. 

        Each query should capture a different aspect or phrasing of the same information need.
        Avoid redundancy, keep them short and clear.
        User question: {question}
        Provide the 3 reformulated search queries, each on a new line:
        """
    ),
)

# Create MultiQueryRetriever，prompt and LLM
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=hybrid_retriever,
    llm=chat_model,
    prompt=mq_prompt,
    include_original=True,
)


In [17]:
import pandas as pd
import time
from ragas import EvaluationDataset

def get_eval_ds(path, retriever, rag_chain):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rs_times = []
    dataset = []

    for query, reference in zip(queries, references):

        
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        # measure response time
        t0 = time.perf_counter()
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rs_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rs_times

In [18]:
from ragas import evaluate, RunConfig
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    run_config = RunConfig(
    timeout=600,      
    max_workers=2,    
    max_retries=2,    
    max_wait=600,     
)
    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=run_config,
        )
    
    return result

In [19]:
metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         AnswerRelevancy(),
         Faithfulness(),]

Create final retriever

In [20]:
from FlagEmbedding import FlagReranker

local_path = "hf_models/rerank/bge-reranker-large"

reranker_model = FlagReranker(
    local_path,  
    use_fp16=True,
    devices='cuda:0'
)

In [21]:
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import DocumentCompressorPipeline

reranker_model = HuggingFaceCrossEncoder(model_name="hf_models/rerank/bge-reranker-large",
                                         model_kwargs={"device": "cuda:0"},)
redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding, similarity_threshold=0.9)

final_rt_lst = []
k_values = [5, 10, 15]
for k in k_values:
    reranker  = CrossEncoderReranker(model=reranker_model,top_n=k)

    pipeline_compressor = DocumentCompressorPipeline(
        transformers=[redundant_filter,reranker]
    )

    compression_retriever = ContextualCompressionRetriever( 
        base_compressor=pipeline_compressor,
        base_retriever=mq_retriever
    )
    final_rt_lst.append(compression_retriever)

In [ ]:
import time
import pandas as pd
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser

template = """You are a helpful assistant specializing in 3D printing operations.
Use the following pieces of retrieved context to answer the question.
If you don’t know the answer, just say that you don’t know.
Use two sentences maximum and keep the answer concise.

Question: {question}
Context: {context}
Answer:
"""
prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

rag_chains = []
for retriever in final_rt_lst:
#  Build the RAG + reranker chain
    rag_chain = (
        {
            "context": retriever | RunnableLambda(format_docs),
            "question": RunnablePassthrough()
        }
        | prompt
        | chat_model
        | StrOutputParser()
    )
    rag_chains.append(rag_chain)

In [23]:
rag_chain.invoke("What is extruder blob in 3D printing?")

`generation_config` default values have been modified to match model-specific defaults: {'do_sample': True}. If this is not desired, please set these values explicitly.


'An extruder blob, also called a "blob of doom," is a large accumulation of filament that sticks to the nozzle and hotend of a 3D printer, typically forming when the first layer detaches from the print bed and adheres to the moving nozzle while extruding.'

test and save results

In [24]:
data_path = "evaluate_dataset/test_dataset.csv"

results_lst = []

for retriever, rag_chain in zip(final_rt_lst, rag_chains):
    eval_ds, response_time = get_eval_ds(data_path, retriever, rag_chain)
    result = get_eval_result(eval_ds, metrics)
    results_lst.append((result,response_time))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Evaluating: 100%|██████████| 200/200 [22:08<00:00,  6.64s/it]


In [ ]:
for idx, (result, response_time) in enumerate(results_lst):
    print(f'top-{k_values[idx]}:', result)
    df = result.to_pandas()
    df["response_time"] = response_time
    df.to_csv(f"./evaluate_results/07_model_test/Qwen3-4B-Instruct-2507/top_{k_values[idx]}.csv")

top-5: {'llm_context_precision_with_reference': 0.6889, 'context_recall': 0.5889, 'nv_accuracy': 0.4750, 'answer_relevancy': 0.9032, 'faithfulness': 0.6559}
top-10: {'llm_context_precision_with_reference': 0.7158, 'context_recall': 0.6349, 'nv_accuracy': 0.5000, 'answer_relevancy': 0.9304, 'faithfulness': 0.7933}
top-15: {'llm_context_precision_with_reference': 0.6797, 'context_recall': 0.7054, 'nv_accuracy': 0.4938, 'answer_relevancy': 0.9036, 'faithfulness': 0.8172}


GPT-4

In [26]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [27]:
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=hybrid_retriever,
    llm=chat_model,
    prompt=mq_prompt,
    include_original=True,
)

In [28]:
for k in k_values:
    reranker  = CrossEncoderReranker(model=reranker_model,top_n=k)

    pipeline_compressor = DocumentCompressorPipeline(
        transformers=[redundant_filter,reranker]
    )

    compression_retriever = ContextualCompressionRetriever( 
        base_compressor=pipeline_compressor,
        base_retriever=mq_retriever
    )
    final_rt_lst.append(compression_retriever)

In [ ]:
rag_chains = []
for retriever in final_rt_lst:
#  Build the RAG + reranker chain
    rag_chain = (
        {
            "context": retriever | RunnableLambda(format_docs),
            "question": RunnablePassthrough()
        }
        | prompt
        | chat_model
        | StrOutputParser()
    )
    rag_chains.append(rag_chain)

In [30]:
rag_chain.invoke("What is extruder blob in 3D printing?")

'An extruder blob, also known as a "blob of doom," is a large accumulation of filament around the nozzle and hotend that occurs when a print detaches from the bed and sticks to the nozzle while continuing to extrude. This issue can be challenging to remove and may require heating the blob to soften it for removal.'

In [ ]:
data_path = "evaluate_dataset/test_dataset.csv"

results_lst = []

for retriever, rag_chain in zip(final_rt_lst, rag_chains):
    eval_ds, response_time = get_eval_ds(data_path, retriever, rag_chain)
    result = get_eval_result(eval_ds, metrics)
    results_lst.append((result,response_time))

In [ ]:
for idx, (result, response_time) in enumerate(results_lst):
    print(f'top-{k_values[idx]}:', result)
    df = result.to_pandas()
    df["response_time"] = response_time
    df.to_csv(f"./evaluate_results/07_model_test/GPT-4o-mini/top_{k_values[idx]}.csv")

top-5: {'llm_context_precision_with_reference': 0.6983, 'context_recall': 0.5517, 'nv_accuracy': 0.4750, 'answer_relevancy': 0.9408, 'faithfulness': 0.6348}
top-10: {'llm_context_precision_with_reference': 0.6911, 'context_recall': 0.6336, 'nv_accuracy': 0.4875, 'answer_relevancy': 0.9197, 'faithfulness': 0.7361}


Llama-3.1-8B-Instruct

In [ ]:
chat = load_llm(model_id="meta-llama/Llama-3.1-8B-Instruct",
                model_path="./hf_models/Llama-3.1-8B-Instruct")

In [ ]:
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=hybrid_retriever,
    llm=chat_model,
    prompt=mq_prompt,
    include_original=True,
)

In [ ]:
for k in k_values:
    reranker  = CrossEncoderReranker(model=reranker_model,top_n=k)

    pipeline_compressor = DocumentCompressorPipeline(
        transformers=[redundant_filter,reranker]
    )

    compression_retriever = ContextualCompressionRetriever( 
        base_compressor=pipeline_compressor,
        base_retriever=mq_retriever
    )
    final_rt_lst.append(compression_retriever)

In [ ]:
rag_chains = []
for retriever in final_rt_lst:
#  Build the RAG + reranker chain
    rag_chain = (
        {
            "context": retriever | RunnableLambda(format_docs),
            "question": RunnablePassthrough()
        }
        | prompt
        | chat_model
        | StrOutputParser()
    )
    rag_chains.append(rag_chain)

In [ ]:
rag_chain.invoke("What is extruder blob in 3D printing?")

In [ ]:
data_path = "evaluate_dataset/test_dataset.csv"

results_lst = []

for retriever, rag_chain in zip(final_rt_lst, rag_chains):
    eval_ds, response_time = get_eval_ds(data_path, retriever, rag_chain)
    result = get_eval_result(eval_ds, metrics)
    results_lst.append((result,response_time))